# Custom Trainign Loop

### Set Up the environment

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.datasets import mnist
import numpy as np

## 1- Basic custom training loop

In [2]:
(x_train , y_train),(x_test , y_test) = mnist.load_data()
x_train, x_test = x_train/255.0 , x_test/255.0
train_dataset = tf.data.Dataset.from_tensor_slices((x_train,y_train)).batch(32)

### Define the model

In [4]:
model = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(128, activation='relu'),
    Dense(10)
])

C:\Users\ilyass\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Define Loss Function and Optimizer

In [5]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

### Implement the custom training loop

In [9]:
epochs = 2
for epoch in range(epochs):
    print(f'Start of epoch {epoch + 1}')

    for step , (x_batch_train , y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            # Forward pass
            logits = model(x_batch_train, trainable = True)
            # Compute loss
            loss_value = loss(y_batch_train, logits)

        # Compute gradient and update weights
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        # Logging the loss every 200 steps
        if step % 200 == 0:
            print(f'Epoch {epoch + 1} Step {step}: Loss = {loss_value.numpy()}')

Start of epoch 1
Epoch 1 Step 0: Loss = 2.370661497116089
Epoch 1 Step 200: Loss = 0.4161525368690491
Epoch 1 Step 400: Loss = 0.21474596858024597
Epoch 1 Step 600: Loss = 0.17736652493476868
Epoch 1 Step 800: Loss = 0.17233097553253174
Epoch 1 Step 1000: Loss = 0.39968544244766235
Epoch 1 Step 1200: Loss = 0.13517475128173828
Epoch 1 Step 1400: Loss = 0.31251662969589233
Epoch 1 Step 1600: Loss = 0.19151119887828827
Epoch 1 Step 1800: Loss = 0.19365572929382324
Start of epoch 2
Epoch 2 Step 0: Loss = 0.06704534590244293
Epoch 2 Step 200: Loss = 0.1355462670326233
Epoch 2 Step 400: Loss = 0.13713927567005157
Epoch 2 Step 600: Loss = 0.06038065254688263
Epoch 2 Step 800: Loss = 0.0786568894982338
Epoch 2 Step 1000: Loss = 0.22653788328170776
Epoch 2 Step 1200: Loss = 0.04973922669887543
Epoch 2 Step 1400: Loss = 0.1896047741174698
Epoch 2 Step 1600: Loss = 0.15175525844097137
Epoch 2 Step 1800: Loss = 0.09970269352197647


## 2- Adding Accuracy Metric

In [10]:
# Set Up the environment
(x_train , y_train),(x_test,y_test) = mnist.load_data()

# Norm
x_train, x_test = x_train/255.0 , x_test/255.0

# create a batched dataset 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train,y_train)).batch(32)

### Define the Model

In [11]:
model = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(128 , activation='relu'),
    Dense(10)
])

C:\Users\ilyass\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Define the loss function , optimizer and metric

In [13]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
accuracy = tf.keras.metrics.SparseCategoricalAccuracy()

### Implement the custom training loop with accuracy

In [14]:
epochs = 5

for epoch in range(epochs):
    print(f'Start of epoch: {epoch}')

    for step,(x_batch_train,y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train,training=True)
            loss_value = loss(y_batch_train,logits)

        grads = tape.gradient(loss_value,model.trainable_weights)
        optimizer.apply_gradients(zip(grads,model.trainable_weights))

        accuracy.update_state(y_batch_train,logits)

        if step % 200 == 0:
            print(f'Epoch {epoch + 1} step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy.result().numpy()}')

    accuracy.reset_state()

Start of epoch: 0
Epoch 1 step 0: Loss = 2.2745471000671387 Accuracy = 0.0625
Epoch 1 step 200: Loss = 0.43681907653808594 Accuracy = 0.8337997794151306
Epoch 1 step 400: Loss = 0.17046767473220825 Accuracy = 0.8688434958457947
Epoch 1 step 600: Loss = 0.15364888310432434 Accuracy = 0.8826435208320618
Epoch 1 step 800: Loss = 0.1348254233598709 Accuracy = 0.8953261375427246
Epoch 1 step 1000: Loss = 0.40226444602012634 Accuracy = 0.9023164510726929
Epoch 1 step 1200: Loss = 0.2220892310142517 Accuracy = 0.9089300632476807
Epoch 1 step 1400: Loss = 0.25474274158477783 Accuracy = 0.9137000441551208
Epoch 1 step 1600: Loss = 0.20802971720695496 Accuracy = 0.9168097972869873
Epoch 1 step 1800: Loss = 0.14308619499206543 Accuracy = 0.9209640622138977
Start of epoch: 1
Epoch 2 step 0: Loss = 0.08422772586345673 Accuracy = 0.96875
Epoch 2 step 200: Loss = 0.14774969220161438 Accuracy = 0.9586442708969116
Epoch 2 step 400: Loss = 0.1030224934220314 Accuracy = 0.957761824131012
Epoch 2 step 600

## 3- Custom Callback for advance looping

In [15]:
# Set Up the environment
(x_train , y_train),(x_test,y_test) = mnist.load_data()

# Norm
x_train, x_test = x_train/255.0 , x_test/255.0

# create a batched dataset 
train_dataset = tf.data.Dataset.from_tensor_slices((x_train,y_train)).batch(32)

In [16]:
model = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(128 , activation='relu'),
    Dense(10)
])

C:\Users\ilyass\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [17]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
accuracy = tf.keras.metrics.SparseCategoricalAccuracy()

### custom callback

In [18]:
from tensorflow.keras.callbacks import Callback

class CustomCallback(Callback):
    def on_epoch_end(self,epoch,logs=None):
        logs = logs or {}
        print(f'End of epoch {epoch + 1}, loss: {logs.get('loss')}, accuracy: {logs.get('accuracy')}')

In [19]:
epochs = 2
custom_callback = CustomCallback()

for epoch in range(epochs):
    print(f'Start of epoch: {epoch}')

    for step,(x_batch_train,y_batch_train) in enumerate(train_dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch_train,training=True)
            loss_value = loss(y_batch_train,logits)

        grads = tape.gradient(loss_value,model.trainable_weights)
        optimizer.apply_gradients(zip(grads,model.trainable_weights))

        accuracy.update_state(y_batch_train,logits)

        if step % 200 == 0:
            print(f'Epoch {epoch + 1} step {step}: Loss = {loss_value.numpy()} Accuracy = {accuracy.result().numpy()}')

    custom_callback.on_epoch_end(epoch, logs={'loss': loss_value.numpy(), 'accuracy' : accuracy.result().numpy()})            
    accuracy.reset_state()


Start of epoch: 0
Epoch 1 step 0: Loss = 2.4182491302490234 Accuracy = 0.03125
Epoch 1 step 200: Loss = 0.41645485162734985 Accuracy = 0.8361318111419678
Epoch 1 step 400: Loss = 0.17702041566371918 Accuracy = 0.869856595993042
Epoch 1 step 600: Loss = 0.1561947613954544 Accuracy = 0.8850873708724976
Epoch 1 step 800: Loss = 0.17741060256958008 Accuracy = 0.8965746164321899
Epoch 1 step 1000: Loss = 0.44175535440444946 Accuracy = 0.9036900401115417
Epoch 1 step 1200: Loss = 0.1764458268880844 Accuracy = 0.9096846580505371
Epoch 1 step 1400: Loss = 0.20782755315303802 Accuracy = 0.9144138097763062
Epoch 1 step 1600: Loss = 0.2032938003540039 Accuracy = 0.9176686406135559
Epoch 1 step 1800: Loss = 0.21571359038352966 Accuracy = 0.9216927886009216
End of epoch 1, loss: 0.03301340341567993, accuracy: 0.9236999750137329
Start of epoch: 1
Epoch 2 step 0: Loss = 0.07222862541675568 Accuracy = 1.0
Epoch 2 step 200: Loss = 0.16518834233283997 Accuracy = 0.9612873196601868
Epoch 2 step 400: Loss

## 4- Add Hidden layers

In [20]:
input_layer = Input(shape=(28,28))

hidden_layer1 = Dense(64, activation='relu')(input_layer)
hidden_layer2 = Dense(64, activation='relu')(hidden_layer1)

In [21]:
output_layer = Dense(1,activation='sigmoid')(hidden_layer2)

In [22]:
model = Model(inputs = input_layer, outputs = output_layer)

In [23]:
model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

In [27]:
model = Sequential([
    Flatten(input_shape=(20,)),
    Dense(128 , activation='relu'),
    Dense(1 , activation='sigmoid')    
])

model.compile(optimizer='adam',loss='binary_crossentropy', metrics=['accuracy'])

x_train = np.random.rand(1000,20)
y_train = np.random.randint(2,size=(1000,1))

model.fit(x_train,y_train,epochs=10,batch_size=32)

Epoch 1/10


C:\Users\ilyass\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4980 - loss: 0.7013
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5110 - loss: 0.6925
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5490 - loss: 0.6883
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5560 - loss: 0.6867
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5650 - loss: 0.6828
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5710 - loss: 0.6827
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5650 - loss: 0.6826
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5720 - loss: 0.6827
Epoch 9/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5820 - loss: 0.6779
Epoch 10/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5700 - loss: 0.6782


In [30]:
x_test = np.random.rand(200,20)
y_test = np.random.randint(2,size=(200,1))
loss , accuracy = model.evaluate(x_test, y_test)
print(f'Test loss: {loss}')
print(f'Test accuracy: {accuracy}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4350 - loss: 0.7118 
Test loss: 0.7117785811424255
Test accuracy: 0.4350000023841858
